# 02 · UNet: encontrar la hoja en la foto

La UNet mira una foto y dice, **píxel por píxel**, si es hoja o fondo.

- Aprende con las máscaras automáticas del notebook 01.
- Se evalúa con las 50 máscaras corregidas a mano.

Se entrena en **Google Colab con GPU**. Antes, subir a `Mi unidad/Proyecto_Final_IA/` los archivos
`utils.py`, este notebook y `datos/dataset_unet.zip`.

## 1. Preparar el entorno (Colab o compu)

In [ ]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.insert(0, '/content/drive/MyDrive/Proyecto_Final_IA')

import utils
entorno = utils.configurar_entorno(['datos/dataset_unet'])
ROOT, DATA = entorno['ROOT'], entorno['DATA']

## 2. Librerías y configuración

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from skimage.io import imread
from tensorflow.keras.layers import Input, Rescaling, Conv2D, Dropout, MaxPooling2D, Conv2DTranspose, concatenate
from tensorflow.keras.models import Model

utils.set_estilo()                           # letra grande en los gráficos
tf.keras.utils.set_random_seed(utils.SEED)   # mismos resultados cada vez

SMOKE = os.environ.get('SMOKE') == '1'       # prueba rápida en la compu

width_shape, height_shape = 160, 160
batch_size = 16
epochs = 1 if SMOKE else 50                  # máximo: corta solo si deja de mejorar

data_path = DATA / 'datos' / 'dataset_unet'

## 3. Cargar los datos
Fotos y máscaras tienen el **mismo nombre**, así cada foto queda con su máscara.
Las máscaras pasan de 0/255 a 0/1, que es lo que responde la red.

In [ ]:
def cargar(carpeta, carpeta_masks='masks'):
    nombres = sorted(os.listdir(data_path / carpeta / 'images'))
    if SMOKE:
        nombres = nombres[:20]
    X, Y = [], []
    for nombre in nombres:
        X.append(imread(data_path / carpeta / 'images' / nombre))
        Y.append(imread(data_path / carpeta / carpeta_masks / nombre))
    X = np.asarray(X, dtype=np.uint8)                          # (N, 160, 160, 3)
    Y = (np.asarray(Y) > 127).astype(np.float32)[..., None]    # (N, 160, 160, 1), 0 y 1
    return X, Y, nombres

X_train, Y_train, _ = cargar('train')
X_val, Y_val, _ = cargar('val')
X_test, Y_test, nombres_test = cargar('test')          # solo para evaluar al final
_, Y_truco, _ = cargar('test', 'masks_auto')            # máscaras automáticas, para comparar

print('train:', X_train.shape, '| val:', X_val.shape, '| test:', X_test.shape)

In [ ]:
plt.imshow(X_train[0])
plt.show()
plt.imshow(np.squeeze(Y_train[0]), cmap='gray')
plt.show()

## 4. La UNet (con Dropout)

In [ ]:
Image_input = Input((height_shape, width_shape, 3))
inputs = Rescaling(1/255)(Image_input)   # pasa los píxeles de 0-255 a 0-1

# camino de bajada
conv1 = Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
conv1 = Dropout(0.1)(conv1)
conv1 = Conv2D(64, (3, 3), activation='relu', padding='same')(conv1)
maxp1 = MaxPooling2D((2, 2))(conv1)

conv2 = Conv2D(128, (3, 3), activation='relu', padding='same')(maxp1)
conv2 = Dropout(0.1)(conv2)
conv2 = Conv2D(128, (3, 3), activation='relu', padding='same')(conv2)
maxp2 = MaxPooling2D((2, 2))(conv2)

conv3 = Conv2D(256, (3, 3), activation='relu', padding='same')(maxp2)
conv3 = Dropout(0.2)(conv3)
conv3 = Conv2D(256, (3, 3), activation='relu', padding='same')(conv3)
maxp3 = MaxPooling2D((2, 2))(conv3)

conv4 = Conv2D(512, (3, 3), activation='relu', padding='same')(maxp3)
conv4 = Dropout(0.2)(conv4)
conv4 = Conv2D(512, (3, 3), activation='relu', padding='same')(conv4)
maxp4 = MaxPooling2D((2, 2))(conv4)

conv5 = Conv2D(1024, (3, 3), activation='relu', padding='same')(maxp4)
conv5 = Dropout(0.3)(conv5)
conv5 = Conv2D(1024, (3, 3), activation='relu', padding='same')(conv5)

# camino de subida
up6 = Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(conv5)
up6 = concatenate([up6, conv4])
conv6 = Conv2D(512, (3, 3), activation='relu', padding='same')(up6)
conv6 = Dropout(0.2)(conv6)
conv6 = Conv2D(512, (3, 3), activation='relu', padding='same')(conv6)

up7 = Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(conv6)
up7 = concatenate([up7, conv3])
conv7 = Conv2D(256, (3, 3), activation='relu', padding='same')(up7)
conv7 = Dropout(0.2)(conv7)
conv7 = Conv2D(256, (3, 3), activation='relu', padding='same')(conv7)

up8 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(conv7)
up8 = concatenate([up8, conv2])
conv8 = Conv2D(128, (3, 3), activation='relu', padding='same')(up8)
conv8 = Dropout(0.1)(conv8)
conv8 = Conv2D(128, (3, 3), activation='relu', padding='same')(conv8)

up9 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(conv8)
up9 = concatenate([up9, conv1])
conv9 = Conv2D(64, (3, 3), activation='relu', padding='same')(up9)
conv9 = Dropout(0.1)(conv9)
conv9 = Conv2D(64, (3, 3), activation='relu', padding='same')(conv9)

outputs = Conv2D(1, (1, 1), activation='sigmoid')(conv9)

model = Model(inputs=[Image_input], outputs=[outputs])
model.compile(optimizer='adam', loss='binary_crossentropy',
              metrics=[tf.keras.metrics.BinaryIoU(target_class_ids=[1], name='iou')])

## 5. Entrenar
- **IoU** en vez de accuracy: si la red dijera "todo es fondo", el accuracy igual daría alto; el IoU daría 0.
- **EarlyStopping:** si pasan 6 épocas sin mejorar en val, corta y se queda con la mejor.

In [ ]:
parar = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)

results = model.fit(X_train, Y_train, validation_data=(X_val, Y_val),
                    batch_size=batch_size, epochs=epochs, callbacks=[parar])

(ROOT / 'models').mkdir(exist_ok=True)
model.save(ROOT / 'models' / 'unet.keras')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
ax[0].plot(results.history['loss'], label='train')
ax[0].plot(results.history['val_loss'], label='val')
ax[0].set_title('Pérdida')
ax[0].legend()
ax[1].plot(results.history['iou'], label='train')
ax[1].plot(results.history['val_iou'], label='val')
ax[1].set_title('IoU')
ax[1].legend()
utils.guardar_figura(fig, 'unet_curvas')
plt.show()
print('Épocas entrenadas:', len(results.history['loss']))

## 6. El examen: IoU en las 50 fotos de test
Comparamos la UNet y las máscaras automáticas contra las corregidas a mano.
IoU = parte donde coinciden / todo lo que marcó alguna de las dos (1 = perfecto, 0 = nada).

In [ ]:
def iou(verdad, prediccion):
    verdad = verdad > 0.5
    prediccion = prediccion > 0.5
    return np.logical_and(verdad, prediccion).sum() / np.logical_or(verdad, prediccion).sum()

preds = model.predict(X_test)

clases = pd.read_csv(data_path / 'lista.csv').set_index('archivo')['clase']
res = pd.DataFrame({
    'clase': [clases[n] for n in nombres_test],
    'UNet': [iou(Y_test[i], preds[i]) for i in range(len(X_test))],
    'Automática': [iou(Y_test[i], Y_truco[i]) for i in range(len(X_test))],
})
print(res.groupby('clase').mean().round(3))
print('\nPromedio:')
print(res[['UNet', 'Automática']].mean().round(3))

In [ ]:
fig, ax = plt.subplots(3, 4, figsize=(16, 12))
una_por_clase = res.groupby('clase').head(1).index
for j, i in enumerate(una_por_clase):
    ax[0, j].imshow(X_test[i])
    ax[1, j].imshow(np.squeeze(Y_test[i]), cmap='gray')
    ax[2, j].imshow(np.squeeze(preds[i]) > 0.5, cmap='gray')
    ax[0, j].set_title(res['clase'][i], fontsize=14)
for a in ax.flat:
    a.axis('off')
ax[1, 0].set_title('a mano', loc='left')
ax[2, 0].set_title('UNet', loc='left')
utils.guardar_figura(fig, 'unet_ejemplos_test')
plt.show()

## 7. Conclusiones
*Completar después de entrenar, solo con los números de arriba.*

- IoU promedio de la UNet: …
- IoU promedio de las máscaras automáticas: …

La UNet aprendió de máscaras automáticas con errores, así que puede repetirlos. Además, el test son solo
50 fotos, casi todas con fondo liso.